In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [15]:
import torch
from transformers import AutoModelForSequenceClassification, AutoConfig
#from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, Trainer
from transformers import BertTokenizer, BertForSequenceClassification, BertConfig
import numpy 
import numpy as np
from scipy.special import softmax
import tensorflow as tf

In [16]:
# Create class for data preparation
class SimpleDataset:
    def __init__(self, tokenized_texts):
        self.tokenized_texts = tokenized_texts
    
    def __len__(self):
        return len(self.tokenized_texts["input_ids"])
    
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.tokenized_texts.items()}

In [17]:
def sentiment_labels(text):
    encoded_input = tokenizer(text, padding=True,truncation=True,max_length=512,return_tensors='pt')
    output = model(**encoded_input)
    # Use .squeeze() to remove the batch dimension safely
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    ranking = np.argsort(scores)
    ranking = ranking[::-1]
    return config.id2label[ranking[0]]

In [18]:
data = [
        "Dr Priya Patel in the respiratory clinic was absolutely wonderful. She took time to explain my COPD management plan and made sure I understood every step. The nurse on reception, Karen Thompson, was also very welcoming.",
        "The parking at 45 Albert Street is very limited. I had to walk from the Westfield car park which is difficult at my age.",
        "I had my appointment on 15 January 2026 at 2pm and wasn't seen until after 3:15pm. Dr James Richardson seemed rushed and didn't listen to my concerns about the medication side effects. I called the clinic at 07 3456 7890 twice before my appointment to ask about preparation and nobody answered. My wife Angela Chen had a much better experience at the Chermside Day Surgery last month.",
        "The physiotherapist Michael was great. He gave me a detailed home exercise program after my knee surgery and followed up via email at susan.obrien82@gmail.com to check on my progress. Very impressed with that level of care.",
        "The online booking system could be easier to use. I ended up calling reception to book because I couldn't figure out the Zedoc portal. I was referred by Dr Nguyen at the Ipswich Hospital and the transition was smooth.",
        "Nurse Rebecca Taylor in the diabetes clinic was exceptional. She spent over 40 minutes with me going through my HbA1c results and adjusting my insulin plan. She also coordinated with my GP Dr Samantha Lee at the Toowoomba Medical Practice to ensure consistent care.",
        "Dr Patel is always thorough and patient. She remembers details from previous visits which makes me feel valued.",
        "I'm 81 years old and find the forms quite difficult to fill in. My daughter Sarah Fitzpatrick usually helps me but she wasn't available this time. Could you offer some assistance at the front desk for elderly patients? Also my Medicare number is 2345 67890 1 and I think there was a billing error on my last visit on 28 January 2026.",
        "The midwifery team, especially Lisa and Jenny, were amazing throughout my pregnancy. They made me feel safe and supported. The antenatal classes at the centre on George Street were also excellent.",
        "It would be nice to have later appointment slots. As a working mum I find it hard to attend before 4pm. My employer at Bright Horizons Childcare is not always flexible with time off. I recommended this clinic to my sister-in-law Patricia Nakamura who is expecting in July.",
        "The new blood collection nurse, Daniel Kim, was very skilled. Best blood draw I've had - barely felt it. He mentioned he previously worked at the Royal Brisbane and Women's Hospital.",
        "The results portal is confusing. I had to call Dr Richardson's office at 07 3456 7891 to get my pathology explained because I couldn't understand the online report.",
        "Dr Patel and Nurse Rebecca were both excellent. They were respectful of my cultural needs and ensured a female practitioner was available for my examination. This was very important to me.",
        "The interpreter service was not available on 4 March 2026 when my mother attended. She speaks Arabic and struggled to communicate her symptoms. Please ensure interpreters are booked in advance. My mother Zahra Al-Rashid (patient ID PAT-91603) would like to provide feedback separately. Can someone contact her at fatima.alrashid@outlook.com to arrange an Arabic-language survey?",
        "Outstanding mental health support from psychologist Dr Amanda Clarke. She helped me develop coping strategies after my workplace incident at BHP Mitsubishi Alliance in Mount Isa last year. The telehealth option made it possible to continue sessions when I was FIFO.",
        "The mental health waiting list is too long. I was referred on 15 November 2025 and didn't get my first appointment until 8 January 2026. That's nearly 8 weeks.",
        "The wound care nurse Jacinta was thorough and gentle. She explained the healing process for my post-surgical wound clearly and gave me written instructions to take home.",
        "I received a reminder SMS from Zedoc to complete this survey but the link didn't work on my Samsung phone. I had to use my husband David Tran's iPhone instead. The SMS came from number 0437 123 456. I was transferred from Logan Hospital after my surgery there on 2 March 2026. The handover between hospitals could have been smoother - my medication list wasn't updated correctly.",
        "The entire cardiology team is first-rate. Dr Richardson personally called me at home on 0478 234 567 to discuss my stress test results. That kind of proactive communication is rare and very reassuring.",
        "The car park needs more disabled bays. I have a temporary mobility permit after my cardiac rehab and struggled to find a spot on 20 March 2026. My cardiologist at the Prince Charles Hospital, Dr Andrew Walsh, also coordinates with Dr Richardson here which gives me great confidence in my care plan."
       ]

In [19]:
# load tokenizer and model, create trainer
MODEL = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
trainer = Trainer(model=model)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
for i, text in enumerate(data):
    print(f" * sentece: {i + 1}: {text}")
    print(">>>", sentiment_labels(text), "\n")

 * sentece: 1: Dr Priya Patel in the respiratory clinic was absolutely wonderful. She took time to explain my COPD management plan and made sure I understood every step. The nurse on reception, Karen Thompson, was also very welcoming.
>>> joy 

 * sentece: 2: The parking at 45 Albert Street is very limited. I had to walk from the Westfield car park which is difficult at my age.
>>> neutral 

 * sentece: 3: I had my appointment on 15 January 2026 at 2pm and wasn't seen until after 3:15pm. Dr James Richardson seemed rushed and didn't listen to my concerns about the medication side effects. I called the clinic at 07 3456 7890 twice before my appointment to ask about preparation and nobody answered. My wife Angela Chen had a much better experience at the Chermside Day Surgery last month.
>>> anger 

 * sentece: 4: The physiotherapist Michael was great. He gave me a detailed home exercise program after my knee surgery and followed up via email at susan.obrien82@gmail.com to check on my prog

In [21]:
# Tokenize texts and create prediction data set
tokenized_texts = tokenizer(data,truncation=True,padding=True)
pred_dataset = SimpleDataset(tokenized_texts)

In [22]:
# Run predictions
predictions = trainer.predict(pred_dataset)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [23]:
# Transform predictions to labels
preds = predictions.predictions.argmax(-1)
labels = pd.Series(preds).map(model.config.id2label)
scores = (np.exp(predictions[0])/np.exp(predictions[0]).sum(-1,keepdims=True)).max(1)

In [24]:
# scores raw
temp = (np.exp(predictions[0])/np.exp(predictions[0]).sum(-1,keepdims=True))

In [25]:
# work in progress
# container
anger = []
disgust = []
fear = []
joy = []
neutral = []
sadness = []
surprise = []

# extract scores (as many entries as exist in pred_texts)
for i in range(len(data)):
  anger.append(temp[i][0])
  disgust.append(temp[i][1])
  fear.append(temp[i][2])
  joy.append(temp[i][3])
  neutral.append(temp[i][4])
  sadness.append(temp[i][5])
  surprise.append(temp[i][6])

In [26]:
# Create DataFrame with texts, predictions, labels, and scores
df = pd.DataFrame(list(zip(data,preds,labels,scores,  anger, disgust, fear, joy, neutral, sadness, surprise)), columns=['text','pred','label','score', 'anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise'])
df.head(20)

,text,pred,label,score,anger,disgust,fear,joy,neutral,sadness,surprise
0,Dr Priya Patel in the respiratory clinic was a...,3,joy,0.893033,0.002067,0.017970,0.003182,0.893033,0.071016,0.004036,0.008695
1,The parking at 45 Albert Street is very limite...,4,neutral,0.357574,0.023155,0.061973,0.249478,0.003733,0.357574,0.208463,0.095626
2,I had my appointment on 15 January 2026 at 2pm...,0,anger,0.888806,0.888806,0.004911,0.032059,0.000964,0.022469,0.024909,0.025881
3,The physiotherapist Michael was great. He gave...,6,surprise,0.922387,0.004140,0.007233,0.001706,0.004315,0.057647,0.002573,0.922387
4,The online booking system could be easier to u...,4,neutral,0.534538,0.008369,0.006630,0.021064,0.318241,0.534538,0.060814,0.050344
5,Nurse Rebecca Taylor in the diabetes clinic wa...,4,neutral,0.891067,0.006766,0.018203,0.005286,0.041374,0.891067,0.007351,0.029954
6,Dr Patel is always thorough and patient. She r...,3,joy,0.946568,0.006360,0.010991,0.000696,0.946568,0.029498,0.003969,0.001917
7,I'm 81 years old and find the forms quite diff...,5,sadness,0.555837,0.009571,0.014376,0.024889,0.002699,0.325686,0.555837,0.066943
8,"The midwifery team, especially Lisa and Jenny,...",3,joy,0.965407,0.000757,0.002584,0.002191,0.965407,0.020428,0.005866,0.002767
9,It would be nice to have later appointment slo...,4,neutral,0.446354,0.007266,0.004923,0.021086,0.195352,0.446354,0.266286,0.058732
